# Embedding MRL — Colab runner

Chọn một method và model bên dưới rồi chạy **Runtime → Run all**. Notebook luôn đồng bộ branch mới nhất trước khi train, dùng `data/train/final_data.csv`, chỉ evaluate một lần trên test sau training, và ghi artifacts trực tiếp vào Google Drive.

> Với pair-classification, threshold được tune trực tiếp trên chính test set theo evaluator của repo. Đây là protocol được yêu cầu, nhưng cần mô tả rõ trong paper vì kết quả không phải held-out threshold selection.


## 1. Settings

In [ ]:
#@title Experiment settings { display-mode: "form" }

METHOD = "gsr"  #@param ["mrl", "ese", "mipic", "gsr"]
MODEL = "bert"  #@param ["bert", "tinybert_6l", "bgem3", "qwen3_0.6b"]

EPOCHS = 5  #@param {type:"integer"}
BATCH_SIZE = 128  #@param {type:"integer"}
EVAL_BATCH_SIZE = 64  #@param {type:"integer"}
LEARNING_RATE = 2e-5  #@param {type:"number"}
MAX_LENGTH = 256  #@param {type:"integer"}
SEED = 42  #@param {type:"integer"}
FP16 = False  #@param {type:"boolean"}
MAX_GRAD_NORM = 0.0  #@param {type:"number"}

# Chỉ dùng khi METHOD=gsr. Phải nhỏ hơn EPOCHS nếu GSR_WEIGHT > 0.
GSR_WEIGHT = 1.0  #@param {type:"number"}
GSR_WARMUP_EPOCHS = 1  #@param {type:"integer"}
GSR_TEACHER_BATCH_SIZE = 64  #@param {type:"integer"}

REPO_URL = "https://github.com/duncan-nguyen/embedding-mrl.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
RUN_NAME = ""  #@param {type:"string"}

# BGE-M3/Qwen3 thường cần batch 4–8 trên GPU 16 GB.
assert EPOCHS > 0 and BATCH_SIZE > 0 and EVAL_BATCH_SIZE > 0
if METHOD == "gsr" and GSR_WEIGHT > 0:
    assert 0 <= GSR_WARMUP_EPOCHS < EPOCHS, (
        "GSR_WARMUP_EPOCHS phải nằm trong [0, EPOCHS)."
    )
print(f"Selected: {METHOD.upper()} / {MODEL}")

## 2. Mount Drive, sync code mới nhất và cài package

In [ ]:
import os
import json
import re
import shutil
import subprocess
import sys
import tempfile
import urllib.parse
import urllib.request
import zipfile
from datetime import datetime
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/[Research Space]/[ICLR] Embedding MRL")
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
default_name = f"{METHOD}_{MODEL}_{timestamp}"
base_name = re.sub(r"[^A-Za-z0-9._-]+", "_", RUN_NAME.strip() or default_name)
safe_name = base_name
suffix = 1
while (DRIVE_ROOT / safe_name).exists():
    safe_name = f"{base_name}_{suffix:02d}"
    suffix += 1
RUN_DIR = DRIVE_ROOT / safe_name
RUN_DIR.mkdir(parents=True, exist_ok=False)
CONSOLE_LOG = RUN_DIR / "console.log"
REPO_DIR = Path("/content/embedding-mrl-runtime")

def run(command, *, cwd=None, log_path=CONSOLE_LOG):
    command = [str(part) for part in command]
    print("$", " ".join(command))
    with Path(log_path).open("a", encoding="utf-8") as log:
        log.write("\n$ " + " ".join(command) + "\n")
        process = subprocess.Popen(
            command, cwd=str(cwd) if cwd else None,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1, env={**os.environ, "PYTHONUNBUFFERED": "1"},
        )
        for line in process.stdout:
            print(line, end="")
            log.write(line)
            log.flush()
        return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)

def download_public_snapshot(repo_url, branch, destination):
    match = re.fullmatch(
        r"https://github\.com/([^/]+)/([^/]+?)(?:\.git)?/?", repo_url.strip()
    )
    if not match:
        raise ValueError("REPO_URL phải có dạng https://github.com/owner/repo.git")
    owner, repo = match.groups()
    encoded_branch = urllib.parse.quote(branch, safe="")
    api_url = f"https://api.github.com/repos/{owner}/{repo}/commits/{encoded_branch}"
    headers = {"Accept": "application/vnd.github+json", "User-Agent": "embedding-mrl-colab"}

    print(f"$ GET {api_url}")
    with urllib.request.urlopen(
        urllib.request.Request(api_url, headers=headers), timeout=60
    ) as response:
        commit = json.load(response)["sha"]

    archive_url = f"https://codeload.github.com/{owner}/{repo}/zip/{commit}"
    print(f"$ GET {archive_url}")
    with tempfile.TemporaryDirectory(prefix="embedding-mrl-", dir="/content") as temp_dir:
        temp_dir = Path(temp_dir)
        archive_path = temp_dir / "source.zip"
        with urllib.request.urlopen(
            urllib.request.Request(archive_url, headers=headers), timeout=180
        ) as response, archive_path.open("wb") as output:
            shutil.copyfileobj(response, output)
        with zipfile.ZipFile(archive_path) as archive:
            archive.extractall(temp_dir)
        extracted = [path for path in temp_dir.iterdir() if path.is_dir()]
        if len(extracted) != 1:
            raise RuntimeError(f"Archive GitHub không hợp lệ: {extracted}")
        if destination.exists():
            shutil.rmtree(destination)
        shutil.move(str(extracted[0]), str(destination))
    return commit

# Git transport trong một số Colab runtime có thể giữ credential/proxy lỗi.
# Tải snapshot public theo SHA để không cần git và vẫn tái lập được thí nghiệm.
commit = download_public_snapshot(REPO_URL, BRANCH, REPO_DIR)

run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], cwd=REPO_DIR)
(RUN_DIR / "git_commit.txt").write_text(commit + "\n", encoding="utf-8")
print(f"\nCommit: {commit}")
print(f"Artifacts: {RUN_DIR}")
try:
    run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"])
except (FileNotFoundError, subprocess.CalledProcessError):
    print("WARNING: Không phát hiện GPU. Chọn Runtime → Change runtime type → GPU.")

## 3. Resolve config

Các protocol quan trọng được khóa tại đây: full `final_data.csv`, test split, evaluation bật, và `eval.every_epoch=false`.

In [ ]:
CONFIG_PATH = REPO_DIR / "configs" / METHOD / f"{MODEL}.yaml"
if not CONFIG_PATH.exists():
    available = sorted(
        str(path.relative_to(REPO_DIR))
        for path in (REPO_DIR / "configs").glob("*/*.yaml")
    )
    raise FileNotFoundError(f"Không có {CONFIG_PATH}. Available: {available}")

def yaml_value(value):
    if isinstance(value, bool):
        return "true" if value else "false"
    if value is None:
        return "null"
    if isinstance(value, float):
        text = repr(value)
        if "e" in text.lower():
            mantissa, exponent = text.lower().split("e")
            if "." not in mantissa:
                mantissa += ".0"
            if not exponent.startswith(("+", "-")):
                exponent = "+" + exponent
            text = f"{mantissa}e{exponent}"
        return text
    return str(value)

overrides = [
    f"name={safe_name}",
    f"train.output_dir={RUN_DIR}",
    f"train.epochs={EPOCHS}",
    f"train.batch_size={BATCH_SIZE}",
    f"train.lr={yaml_value(float(LEARNING_RATE))}",
    f"train.seed={SEED}",
    f"train.fp16={yaml_value(FP16)}",
    f"train.max_grad_norm={yaml_value(MAX_GRAD_NORM if MAX_GRAD_NORM > 0 else None)}",
    f"data.max_length={MAX_LENGTH}",
    "data.train_file=train/final_data.csv",
    "data.max_train_samples=null",
    "eval.enabled=true",
    "eval.split=test",
    "eval.every_epoch=false",
    f"eval.batch_size={EVAL_BATCH_SIZE}",
]
if METHOD == "gsr":
    overrides.extend([
        f"gsr.weight={yaml_value(float(GSR_WEIGHT))}",
        f"gsr.warmup_epochs={GSR_WARMUP_EPOCHS}",
        f"gsr.teacher_batch_size={GSR_TEACHER_BATCH_SIZE}",
    ])

TRAIN_ARGS = [item for override in overrides for item in ("--set", override)]
print("Locked protocol:")
print("  train data    = data/train/final_data.csv (full)")
print("  eval           = once after training, split=test")
print("  pair threshold = tuned on test by evaluator")
print("  output         =", RUN_DIR)
run(
    [sys.executable, "scripts/train.py", "--config", CONFIG_PATH,
     *TRAIN_ARGS, "--print-config"],
    cwd=REPO_DIR,
)

## 4. Train và evaluate một lần

In [ ]:
import time

started = time.time()
run(
    [sys.executable, "scripts/train.py", "--config", CONFIG_PATH, *TRAIN_ARGS],
    cwd=REPO_DIR,
)
print(f"\nHoàn tất sau {(time.time() - started) / 60:.1f} phút")
print("Console log:", CONSOLE_LOG)
print("Trainer log:", RUN_DIR / "train.log")

## 5. Kết quả

In [ ]:
import json
import pandas as pd
from IPython.display import display

report = json.loads((RUN_DIR / "results.json").read_text(encoding="utf-8"))
scores = pd.read_csv(RUN_DIR / "results.csv").set_index("dim")
print(f"{report['experiment']['method'].upper()} / {report['experiment']['model']}")
print("Commit:", (RUN_DIR / "git_commit.txt").read_text().strip())
print("Run directory:", RUN_DIR)
display(scores)

threshold_rows = []
for task, dimensions in report.get("pair", {}).items():
    for dimension, metrics in dimensions.items():
        threshold_rows.append({
            "task": task, "dimension": dimension,
            "test_tuned_threshold": metrics["best_threshold"],
            "test_accuracy": metrics["accuracy"],
            "test_macro_f1": metrics["f1"],
        })
if threshold_rows:
    print("\nThresholds tuned on the test set:")
    display(pd.DataFrame(threshold_rows))

print("\nSaved files:")
for path in sorted(RUN_DIR.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(RUN_DIR)} ({path.stat().st_size / 1024:.1f} KiB)")